# 06 — Dummy baseline on the selected features

Goal: the floor for the models track — `DummyClassifier(strategy="most_frequent")`
on the K = 40 shortlist that notebook 04 persisted to
`outputs/selected_features.json`.

This notebook does no EDA, feature engineering, or encoding. It loads the shared
artifacts (data, folds, shortlist) and runs a model through the standard pipeline.
Every saved run goes to `outputs/runs/` and appends a row to
`outputs/leaderboard.csv`.


In [ ]:
import pandas as pd

from daa.config import OUTPUTS_DIR
from daa.data import explicit_spec, infer_column_spec, load_train
from daa.experiment import run_experiment
from daa.features.selection import load_selected_features
from daa.splits import get_folds


In [ ]:
X, y = load_train()
folds = get_folds(y)
print(f"real data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")
print(f"{len(folds)} folds, {y.nunique()} classes")


In [ ]:
# The handoff from the feature-selection work. The loader re-checks the column
# and target fingerprints, so a stale or hand-edited shortlist fails loudly.
spec = explicit_spec(infer_column_spec(X))
selection = load_selected_features(OUTPUTS_DIR / "selected_features.json", spec, y)
selected_columns = list(selection.columns)
print(f"k={selection.k}: {selection.metadata.get('decision')}")


## The dumbest possible baseline

`run_experiment` builds the shared pipeline (preprocessor + model), runs it on
the frozen folds, and records the run. `columns=selected_columns` restricts the
model to the shortlist; the preprocessor stays the standard one.

`DummyClassifier` always predicts the majority class. It is the floor: any real
model on the same 40 columns must beat it on macro F1.


In [ ]:
dummy_selected = run_experiment(
    "dummy",
    name=f"dummy_selected_k{selection.k}",
    X=X,
    y=y,
    folds=folds,
    columns=selected_columns,
    feature_set=f"selected_k{selection.k}",
    fit_full=True,
)
pd.Series(dummy_selected.metrics, name=dummy_selected.name)


## Where the run went

The two rows that matter are this run and notebook 04's `hist_gb_selected_k40`,
both on the selected K = 40 columns: their gap on macro F1 is the signal the
shortlist carries over the majority class.


In [ ]:
leaderboard = pd.read_csv(OUTPUTS_DIR / "leaderboard.csv")
score_columns = ["run_id", "name", "n_features", "macro_f1_mean", "macro_f1_std"]
leaderboard[leaderboard["name"].str.contains("dummy|selected_k", case=False)][
    score_columns
]


## Next

Swap `"dummy"` for a real model from `daa.models.registry` and keep everything
else — same `columns`, same frozen folds — so the leaderboard compares models,
not pipelines. Preprocessing variants live in `daa.features.preprocess`;
encoding experiments belong to the encoding track, not here.
